# Fault Classification — Offline Exploration

This notebook explores the same building blocks the live system uses:
the `SystemSimulator`, the fault physics in `injector/fault_definitions.py`,
and the sliding-window feature builder. We simulate faults, look at the raw
telemetry, train the classifier, and inspect what it learns.

No Kafka required — everything here is offline.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # import the project from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from injector.fault_definitions import CLASSES, FAULTS
from producers.fault_state import ActiveFault
from producers.simulator import SystemSimulator
from shared.schemas import SENSOR_CHANNEL, CHANNELS
from shared import config

CLASSES

## 1. What does a fault look like in the raw telemetry?

We drive the simulator directly (no feature building) to see how each fault
distorts the three channels. The fault is injected at `onset` seconds.

In [ ]:
def raw_series(fault_name, severity=0.9, duration=20.0, onset=10.0, span=45.0, seed=0):
    """Return {channel: (times, values)} for one fault episode."""
    sim = SystemSimulator(np.random.default_rng(seed))
    series = {ch: ([], []) for ch in CHANNELS}
    for sid, channel in SENSOR_CHANNEL.items():
        period = config.SENSOR_PERIODS[sid]
        t = 0.0
        while t <= span:
            active = [ActiveFault(fault_name, onset, severity, duration)] if t >= onset else []
            v = sim.sample(channel, t, active)
            if v is not None:
                series[channel][0].append(t)
                series[channel][1].append(v)
            t += period
    return series

faults_to_show = [f for f in CLASSES if f != 'NORMAL']
fig, axes = plt.subplots(len(faults_to_show), 1, figsize=(10, 2.2 * len(faults_to_show)), sharex=True)
for ax, fault in zip(axes, faults_to_show):
    s = raw_series(fault)
    for ch in CHANNELS:
        ax.plot(s[ch][0], s[ch][1], label=ch, lw=1)
    ax.axvline(10.0, color='k', ls='--', alpha=0.4)
    ax.set_title(fault, loc='left', fontsize=10)
    ax.legend(loc='upper right', fontsize=7, ncol=3)
axes[-1].set_xlabel('time (s)   —   dashed line = fault injected')
plt.tight_layout(); plt.show()

Notice the different signatures: a **step** for temp bias, a **downward ramp**
for a pressure leak, **noisy elevation** for vibration, a **flat line** when a
sensor sticks, and a **gap** (readings simply stop) on dropout. The classifier
has to recover the fault label from these shapes alone.

## 2. Build the labelled feature dataset

`generate_dataset` replays many episodes through the *same* sliding-window
feature builder the live consumer uses, labelling each row with the fault
active at that instant (`NORMAL` before onset).

In [ ]:
from model.generate_training_data import generate_dataset
from streaming.feature_builder import FEATURE_NAMES

df = generate_dataset()
print(df.shape)
df['label'].value_counts()

## 3. Train and evaluate a RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

X, y = df[FEATURE_NAMES].values, df['label'].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                             class_weight='balanced', n_jobs=-1, random_state=42)
clf.fit(Xtr, ytr)
print(classification_report(yte, clf.predict(Xte), zero_division=0))

## 4. Confusion matrix

In [ ]:
labels = [c for c in CLASSES if c in set(y)]
cm = confusion_matrix(yte, clf.predict(Xte), labels=labels)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=8)
plt.colorbar(im, fraction=0.046); plt.title('Confusion matrix'); plt.tight_layout(); plt.show()

## 5. Which features carry the signal?

Feature importances line up with the fault physics: `age`/`count` on pressure
flag dropout, `std` on temperature flags a stuck sensor, `slope`/`min` on
pressure flag a leak, and `std`/`max` on vibration flag a spike.

In [ ]:
imp = pd.Series(clf.feature_importances_, index=FEATURE_NAMES).sort_values()
ax = imp.tail(15).plot.barh(figsize=(8, 6))
ax.set_title('Top 15 feature importances'); plt.tight_layout(); plt.show()

## 6. Time-to-detect

Accuracy alone hides the streaming trade-off. Run `python -m model.evaluate`
(from the repo root) for the full report including **mean time-to-detect** per
fault and the **false-alarm rate**. Step faults are caught in well under a
second; gradual faults such as a pressure leak take a few seconds to become
visible in the feature window — which is exactly the behaviour a
condition-monitoring system needs to be honest about.